# SPHEREx self-calibration: one channel, end to end

A run is configured in Python, with settings objects that are checked when they are built: the
**instrument** (`sc.SPHEREx(3)`: Detector 3, its LVF chunk map of channels x subchannels x
columns, and its band-centre / band-width maps), the **field** (`sc.Field`: one data set, its
folder, and the machine its actions use) and the **recipe** (`sc.Recipe`: the model, how it is
fitted, coadded and summed, and the name of its products). This notebook reprojects (optional),
calibrates one channel of Detector 3, reads the calibration and the mosaic, shows the model the
preset built and the plan of the run, then checks a spectral model against Detector 4 without
solving.

The paths below point at the Detector 3 NEP frames already reprojected on this machine (300 of
them are used, as in the regression gate) and, for the last step, at the NEP Detector 4 field.
Change them, or set the `SELFCAL_DEMO_*` environment variables, for your own run.

In [ ]:
import os
import sys

import numpy as np
import matplotlib.pyplot as plt

REPO = os.path.abspath('..')                       # this notebook lives in <repo>/notebooks/
if REPO not in sys.path:
    sys.path.insert(0, REPO)

import selfcal as sc
from selfcal.geometry.map_helper import chunk_to_det
from selfcal.instruments import spherex

plt.rcParams['figure.dpi'] = 300
sc.set_progress(False)


def show(img, title, ax=None, lo=1, hi=99, cmap='viridis'):
    """An image with its colorbar, clipped to percentiles of its finite values."""
    ax = ax or plt.gca()
    good = img[np.isfinite(img) & (img != 0)]
    vmin, vmax = np.percentile(good, [lo, hi]) if good.size else (None, None)
    im = ax.imshow(img, origin='lower', vmin=vmin, vmax=vmax, cmap=cmap)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_title(title, fontsize=8)
    return im

OUTPUT_DIR = os.environ.get('SELFCAL_DEMO_OUTPUT', '/mnt/md124/thomasli/selfcal/outputs/')
RUN_NAME = 'SPHEREx_nep_qr2_det3_6p2arcsec'           # <OUTPUT_DIR>/<RUN_NAME>/ref.fits = the sky grid
FRAMES = os.environ.get('SELFCAL_DEMO_FRAMES',
                        '/home/thomasli/selfcal-project/selfcal-memopt/cache/reproj_nvme_SPHEREx_nep_qr2_det3_6p2arcsec')
CACHE_DIR = os.environ.get('SELFCAL_DEMO_CACHE', '/tmp/selfcal_demo_cache/')
N_FRAMES = int(os.environ.get('SELFCAL_DEMO_N_FRAMES', 300))
WORKERS = int(os.environ.get('SELFCAL_DEMO_WORKERS', 16))
D4_FIELD = os.environ.get('SELFCAL_DEMO_D4_FIELD',     # a Detector 4 field with its frames (section 6)
                          '/mnt/md124/thomasli/selfcal/outputs/SPHEREx_NEP_2026W17_D4_6p2arcsec')

## 1. The field, and reprojection (once per detector)

`sc.Field(path, instrument, pixel_scale, compute=...)` is one data set: its folder holds the
reference grid `ref.fits`, the frames (`reprojected/`), the products and a record of every action
(`records/`); `sc.Compute` is the machine (scratch disk and worker processes; `stage=None` reads
the frames where they are), which never changes a product. `FIELD.reproject(exposures)` resamples
every exposure onto the reference grid, one frame file each; the frames above already exist, so it
runs only when `REPROJECT = True`, and the calibration then reads the new frames.

In [ ]:
FIELD = sc.Field(os.path.join(OUTPUT_DIR, RUN_NAME), sc.SPHEREx(3, num_col=3), pixel_scale=6.2,
                 compute=sc.Compute(CACHE_DIR, workers=WORKERS, stage=None))
print(FIELD)

REPROJECT = False
if REPROJECT:
    FRAMES = FIELD.reproject('/mnt/md124/SPHEREx/SPHEREx_nep_data/qr2/*/*/*/*D3*.fits', method='exact',
                             padding=100, padding_fraction=0.05, compute=FIELD.compute.replace(workers=50))

## 2. Calibrate one channel

The recipe holds everything that decides the numbers:

* `sc.continuum(...)`: a constant sky per pixel, a free offset per frame and chunk (smoothed along
  the columns with weight 0.1, pulled toward a straight line along them with weight 0.5, zero mean
  in each frame) and a per-frame scalar;
* `sc.Fit`: 50 LSQR iterations and a 5-sigma outlier clip;
* `sc.Coadd`: a 2-sigma clipped mean in which data-quality bit 21 (the source mask) does not flag a
  pixel, each detector pixel sampled 2 x 2 times on the way;
* `sc.Numerics`: the solver's threads (they change the last bits);
* `name`: the products' suffix.

`FIELD.calibrate` solves the job `spherex.channel(17)` on the first `N_FRAMES` frames, builds the
mosaic and returns a `Result`; run again with the same inputs, it reuses the products.

In [ ]:
RECIPE = sc.Recipe(sc.continuum(smooth=0.1, poly_prior=sc.Poly(1, weight=0.5)),
                   fit=sc.Fit(50, clip=5.0),
                   coadd=sc.Coadd(clip=2.0, oversample=2, ignore_flags=[21]),
                   numerics=sc.Numerics(WORKERS),
                   name='notebook_demo')
JOB = spherex.channel(17)
FRAME_FILES = sc.frames_in(FRAMES)[:N_FRAMES]         # the first N_FRAMES frames, read where they are

result = FIELD.calibrate(RECIPE, jobs=JOB, frames=FRAME_FILES)
print(result)

## 3. What the solve produced

`result.cal()` opens the calibration file as a `CalFile`, which reads every schema version: the sky
maps, the per-frame offsets of each chunk map, the per-frame scalars, the frame list.

In [ ]:
with result.cal() as cal:
    print(cal.describe())
    sky = cal.sky('continuum')
    offsets = cal.offsets[0]                 # (frames, chunks)
    scalar = cal.frame_scalar
    chunk_map = cal.chunk_maps[0]            # detector grid -> chunk id

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
show(sky, 'sky: continuum (MJy/sr)', ax=axes[0])
show(chunk_to_det(chunk_map, offsets[0]), 'offset of frame 0 on the detector (MJy/sr)', ax=axes[1],
     cmap='RdBu_r')
axes[2].plot(scalar, '.', ms=2)
axes[2].set_xlabel('frame')
axes[2].set_ylabel('per-frame scalar (MJy/sr)')
plt.tight_layout()

## 4. The mosaic

`result.mosaic()` opens the mosaic as a `MosaicFile`: its maps by name (the mean, the std, the
sigma-clipped mean, the coadded wavelength maps), the WCS and the unit.

In [ ]:
with result.mosaic() as mos:
    print(mos)
    mean, clipped = mos.mean, mos.clipped

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
show(mean, 'mean of the calibrated frames (MJy/sr)', ax=axes[0])
show(clipped, 'sigma-clipped mean (MJy/sr)', ax=axes[1])
plt.tight_layout()

## 5. The recipe is a model

`sc.continuum` is a preset: a function that returns an `sc.Model`, here the same model as the one
spelled out term by term below. Printing a settings object shows the Python that rebuilds it;
`model.spec()` is the model's data form (the `ModelSpec` the solver reads); `FIELD.plan(...)`
checks a run without computing anything (the model against the instrument's geometry, the frames,
each product and whether it would be made or reused).

In [ ]:
model = RECIPE.model
print(model)
print('same as spelled out:', model == sc.Model(
    sky=[sc.Sky('continuum')],
    offsets=[sc.Offsets(smooth=0.1, poly_prior=[sc.Poly(1, weight=0.5)], mean_zero=True)],
    scalar=True))

spec = model.spec()
for term in spec.sky:
    print('sky   ', term)
for term in spec.offset:
    print('offset', term)
print('scalar', spec.scalar)

print(FIELD.plan(RECIPE, jobs=JOB, frames=FRAME_FILES))

## 6. A spectral model on Detector 4

`spherex.line('aromatic', damping=5e-3)` is a sky term: a map times the shipped 3.29 µm aromatic
PAH template, a tabulated function of the wavelength (the instrument's band-centre map).
`sc.spectral` adds it to a constant sky, with an offset that is a degree-2 polynomial along
subchannels 200 to 320 for each column: the model of the production multi-line fits, with one
line. Planning it on a Detector 4 field checks every name against the Detector 4 geometry and
solves nothing; `d4.instrument.geometry(1)` is that geometry, on which the plot evaluates the
line's coefficient through the sky model.

In [ ]:
AROMATIC = spherex.line('aromatic', damping=5e-3)
SPECTRAL = sc.Recipe(sc.spectral([AROMATIC], polynomial=sc.Poly(2, window=range(200, 321))),
                     fit=sc.Fit(300, clip=5.0, ignore_flags=[21], shot_noise_weights=True), coadd=None,
                     name='notebook_demo_aromatic')
d4 = sc.Field(D4_FIELD, sc.SPHEREx(4, num_col=3), pixel_scale=6.2)
plan = d4.plan(SPECTRAL, jobs=spherex.window('Multiline3', subchannels=range(200, 321)))
print(plan)

sky_model = SPECTRAL.model.spec().build_sky_model(d4.instrument.geometry(1))   # the Detector 4 geometry
print([(c.name, c.coefficient.describe() if c.coefficient else '1') for c in sky_model.components])
lam = np.linspace(3.1, 3.6, 400)
g = sky_model.components[1].coefficients({'BC': lam})
plt.figure(figsize=(5, 3))
plt.plot(lam, g)
plt.xlabel('wavelength (µm)')
plt.ylabel('coefficient of the aromatic map')
plt.tight_layout()